# RAG 01｜🌙 睡不著？來聽一首歌曲背後的故事 —— 建立流行音樂知識庫

**114502527 資工二 張紘齊**

這份 Notebook 把我整理的 **36 首流行歌曲故事**（歐美、日本、華語各 12 首）變成向量資料庫，存成 **`rag-data.zip`**，給第二份 Colab（RAG02）的深夜電台 DJ「音痴」使用。

| 項目 | 內容 |
|---|---|
| 知識來源 | 自行上網查證、改寫整理的 36 篇歌曲 TXT（一首歌一篇），**不含歌詞** |
| 每篇內容 | 表頭（歌手、地區、年份、曲風、場合標籤、來源網址）＋ 5 個段落：創作背景／氛圍與情緒／適合場合與用途／歌手趣聞／延伸冷知識 |
| Embedding 模型 | `google/embeddinggemma-2`（與老師範例相同） |
| 向量資料庫 | FAISS `IndexFlatIP`（正規化後內積 = cosine similarity） |
| 輸出 | `rag-data.zip`（`index.faiss`、`documents.json`、`index_doc_ids.json`、`config.json`） |

> 🔑 **和老師範例不同的地方（重點）**
> 1. **文件切塊（chunking）**：老師是「一篇一個向量」；我把每首歌切成 **6 塊**（基本資料＋5 個段落），共 216 個向量。問「適合婚禮的歌」時，會直接命中各首歌的【適合場合與用途】段落，而不是整篇歌曲介紹。
> 2. **每塊都帶歌名**：切開後的片段開頭都寫上〈歌名〉歌手｜段落名，並把「歌名 - 歌手」放進 EmbeddingGemma 的 `title:` 欄位，避免片段失去上下文。
> 3. **更豐富的 metadata**：地區、年份、曲風、場合標籤、氛圍關鍵字、多個來源網址；RAG02 會用「地區」做篩選。
> 4. **視覺化**：用老師的固定套件 matplotlib 畫出資料分布，以及 216 個向量投影到 2D 的分布圖。
> 5. **雲端下載**：可直接從網址下載原始資料，不一定要手動上傳。

**操作方式：由上往下一格一格執行。** GPU 可選，沒有 GPU 用 CPU 也可以（216 個短片段，幾分鐘內完成）。

## 1. 安裝套件

與老師範例相同的固定版本，方便重現。安裝完再執行下一格；如果本工作階段已載入其他版本，請「重新啟動工作階段」後從下一格繼續。

In [ ]:
%pip install -q "sentence-transformers==6.1.0" "transformers==5.20.1" "faiss-cpu==1.15.1"

## 2. 匯入套件

第一格是**老師的固定套件**（`%matplotlib inline`、NumPy、Pandas、Matplotlib），第二格是本作業額外需要的套件。

In [ ]:
%matplotlib inline

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
from pathlib import Path
from datetime import datetime
from zoneinfo import ZoneInfo
from collections import Counter
from urllib.parse import urlparse, parse_qs
import importlib.metadata as package_metadata
import urllib.request
import hashlib
import io
import json
import re
import tempfile
import zipfile

import torch
import faiss
from IPython.display import display
from sentence_transformers import SentenceTransformer
from huggingface_hub import HfApi

## 3. 集中設定

- `RAW_DATA_URL`：原始 36 篇 TXT 打包成的 `songs-raw.zip` 雲端網址。可以直接貼 **Google Drive 的一般分享連結**，程式會自動轉成下載網址；留空就改成手動上傳。
- `EXPORT_ZIP_NAME`：依作業規定，向量資料庫存成 **`rag-data.zip`**。
- `DOCUMENT_TEMPLATE`：EmbeddingGemma 建議的文件格式是 `title: 標題 | text: 內容`。老師的範例因為正文已含標題而填 `none`；我的片段很短，所以把「歌名 - 歌手」放進 `title`，讓每塊都知道自己屬於哪首歌。

In [ ]:
MODEL_ID = "google/embeddinggemma-2"
BATCH_SIZE = 8
EXPECTED_SONG_COUNT = 36          # 歐美、日本、華語各 12 首
EXPORT_ZIP_NAME = "rag-data.zip"  # 作業規定：向量資料庫存成 rag-data.zip

# 原始資料 songs-raw.zip 的雲端分享連結；留空則改成手動上傳。
RAW_DATA_URL = ""

DOCUMENT_TEMPLATE = "title: {title} | text: {text}"
QUERY_TEMPLATE = "task: search result | query: {question}"

SECTION_NAMES = ["創作背景", "氛圍與情緒", "適合場合與用途", "歌手趣聞", "延伸冷知識"]
REGIONS = ["歐美", "日本", "華語"]

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = (
    torch.bfloat16
    if DEVICE == "cuda" and torch.cuda.is_bf16_supported()
    else torch.float32
)
PACKAGE_NAMES = ["torch", "numpy", "sentence-transformers", "transformers", "faiss-cpu", "huggingface-hub"]
PACKAGE_VERSIONS = {name: package_metadata.version(name) for name in PACKAGE_NAMES}
print("裝置：", DEVICE, "／模型精度：", DTYPE)
if DEVICE == "cuda":
    print("GPU：", torch.cuda.get_device_name(0))
display(pd.DataFrame(PACKAGE_VERSIONS.items(), columns=["套件", "版本"]))

## 4. 取得原始歌曲資料

先嘗試從 `RAW_DATA_URL` 下載；沒有設定或下載失敗時，改用手動上傳 `songs-raw.zip`。

和老師的寫法一樣：只解開 TXT，並確認 zip 裡的路徑不會寫到暫存資料夾外面。

In [ ]:
def to_direct_download_url(url):
    """把 Google Drive 的一般分享連結轉成可直接下載的網址；其他網址原樣回傳。"""
    url = url.strip()
    parsed = urlparse(url)
    if "drive.google.com" not in parsed.netloc:
        return url
    match = re.search(r"/d/([A-Za-z0-9_-]{10,})", parsed.path)
    file_id = match.group(1) if match else parse_qs(parsed.query).get("id", [None])[0]
    if not file_id or not re.fullmatch(r"[A-Za-z0-9_-]{10,}", file_id):
        raise ValueError("看不出 Google Drive 檔案 ID，請確認貼上的是檔案的分享連結。")
    return f"https://drive.google.com/uc?export=download&id={file_id}"


def download_zip_bytes(url, timeout=60):
    """下載 zip 並確認內容真的是 zip（雲端權限沒開時，下載到的會是 HTML 登入頁）。"""
    request = urllib.request.Request(to_direct_download_url(url), headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(request, timeout=timeout) as response:
        data = response.read()
    if not data.startswith(b"PK"):
        raise ValueError("下載到的不是 zip 檔；請確認雲端檔案已設為「知道連結的任何人都能檢視」。")
    return data

In [ ]:
archive_bytes, archive_name = None, None
if RAW_DATA_URL.strip():
    try:
        archive_bytes = download_zip_bytes(RAW_DATA_URL)
        archive_name = "songs-raw.zip"
        print("已從網址下載原始資料：", f"{len(archive_bytes):,} bytes")
    except Exception as exc:
        print("下載失敗，改用手動上傳。原因：", exc)

if archive_bytes is None:
    from google.colab import files
    uploaded = files.upload()
    zip_names = [name for name in uploaded if name.lower().endswith(".zip")]
    if len(zip_names) != 1:
        raise ValueError("請一次上傳一個 songs-raw.zip。")
    archive_name = zip_names[0]
    archive_bytes = uploaded[archive_name]

SOURCE_ARCHIVE_SHA256 = hashlib.sha256(archive_bytes).hexdigest()
DATA_DIR = Path(tempfile.mkdtemp(prefix="song_documents_"))

with zipfile.ZipFile(io.BytesIO(archive_bytes)) as archive:
    members = [
        member for member in archive.infolist()
        if not member.is_dir()
        and member.filename.lower().endswith(".txt")
        and not any(part.startswith((".", "__MACOSX")) for part in Path(member.filename).parts)
    ]
    if not members:
        raise ValueError("zip 中找不到 TXT。")
    for member in members:
        target = (DATA_DIR / member.filename).resolve()
        if not target.is_relative_to(DATA_DIR.resolve()):
            raise ValueError(f"zip 包含不適當的檔案路徑：{member.filename}")
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_bytes(archive.read(member))

txt_paths = sorted(
    [p for p in DATA_DIR.rglob("*") if p.is_file() and p.suffix.lower() == ".txt"],
    key=lambda p: p.name,
)
print("資料：", archive_name, "／TXT 數量：", len(txt_paths))
print("、".join(p.stem for p in txt_paths))
if len(txt_paths) != EXPECTED_SONG_COUNT:
    print(f"提醒：本次為 {len(txt_paths)} 篇，與預期的 {EXPECTED_SONG_COUNT} 篇不同；仍可繼續。")

## 5. 先看一篇原始文件

每篇的格式由我自己設計：前面是「欄位：內容」的表頭，後面是 5 個【段落】。檔名開頭的 `W`／`J`／`C` 代表歐美／日本／華語，後面兩位數是編號，之後就當成歌曲編號使用。

In [ ]:
SAMPLE_INDEX = 0
sample_path = txt_paths[SAMPLE_INDEX]
sample_raw = sample_path.read_text(encoding="utf-8-sig")
print("檔名：", sample_path.name)
print("=" * 60)
print(sample_raw)

## 6. 解析表頭 → metadata

`extract_metadata()` 讀取第一個【段落】之前的表頭。metadata 是用來**顯示、篩選與追溯來源**的資訊，例如 RAG02 的「地區篩選」和「來源連結」都靠它。

- 場合標籤、氛圍關鍵字用「、」分隔，存成清單。
- 來源網址用「;」分隔，一首歌可以有多個來源。
- 找不到的欄位存 `None`，不自行捏造。

In [ ]:
def normalize_text(raw_text):
    return raw_text.lstrip("﻿").replace("\r\n", "\n").replace("\r", "\n")


def read_header(raw_text):
    """讀取第一個【段落】之前、格式為「欄位：內容」的表頭。"""
    header_part = re.split(r"^【", normalize_text(raw_text), maxsplit=1, flags=re.MULTILINE)[0]
    fields = {}
    for line in header_part.splitlines():
        match = re.match(r"^([^：:\n]{1,12})[：:][ \t]*(.*)$", line)
        if match:
            fields[match.group(1).strip()] = match.group(2).strip()
    return fields


def split_list(value, pattern=r"[、,，]"):
    return [item.strip() for item in re.split(pattern, value or "") if item.strip()]


def extract_metadata(raw_text, filename):
    fields = read_header(raw_text)
    year_match = re.match(r"\d{4}", fields.get("發行年份", ""))
    return {
        "song_id": Path(filename).stem.split("_")[0],
        "title": fields.get("標題") or Path(filename).stem,
        "artist": fields.get("歌手") or "未標示",
        "region": fields.get("地區") or "未標示",
        "year": int(year_match.group()) if year_match else None,
        "year_note": fields.get("發行年份"),
        "language": fields.get("語言"),
        "genre": fields.get("曲風"),
        "tags": split_list(fields.get("場合標籤")),
        "moods": split_list(fields.get("氛圍關鍵字")),
        "source_urls": split_list(fields.get("來源網址"), pattern=r"\s*;\s*"),
        "date": fields.get("資料整理日期"),
        "document_type": fields.get("資料性質") or "未標示",
        "filename": filename,
    }


sample_metadata = extract_metadata(sample_raw, sample_path.name)
print(json.dumps(sample_metadata, ensure_ascii=False, indent=2))

## 7. 文件切塊（chunking）—— 本作業的重點修改

老師的範例是一篇文件一個向量。歌曲故事有很多不同面向，如果整篇做成一個向量，問「哪首歌適合畢業典禮？」時，「創作背景」「紀錄」等不相關的內容會把向量稀釋掉。

所以 `build_chunks()` 把**每首歌切成 6 塊**：

| 片段 | 內容 | 片段編號範例 |
|---|---|---|
| 0 | 基本資料（歌手、地區、年份、曲風、場合標籤、氛圍） | `W01-0` |
| 1～5 | 創作背景／氛圍與情緒／適合場合與用途／歌手趣聞／延伸冷知識 | `W01-1`～`W01-5` |

每塊開頭都會加上「〈歌名〉歌手｜段落名」，切開後仍然知道在講哪首歌；LLM 回答時也用這個片段編號標註出處。

In [ ]:
SECTION_PATTERN = re.compile(
    r"^【(?P<name>[^】\n]+)】[ \t]*\n(?P<body>.*?)(?=^【[^】\n]+】|\Z)",
    flags=re.MULTILINE | re.DOTALL,
)


def extract_sections(raw_text):
    sections = {
        match.group("name").strip(): match.group("body").strip()
        for match in SECTION_PATTERN.finditer(normalize_text(raw_text))
    }
    missing = [name for name in SECTION_NAMES if not sections.get(name)]
    if missing:
        raise ValueError(f"缺少段落：{missing}")
    return sections


def build_chunks(raw_text, metadata):
    """一首歌切成 6 塊：基本資料＋5 個段落；每塊開頭都寫上歌名與段落名。"""
    sections = extract_sections(raw_text)
    heading = f"〈{metadata['title']}〉{metadata['artist']}"
    profile = "\n".join([
        f"歌手：{metadata['artist']}",
        f"地區：{metadata['region']}",
        f"發行年份：{metadata['year_note'] or '未提供'}",
        f"語言：{metadata['language'] or '未提供'}",
        f"曲風：{metadata['genre'] or '未提供'}",
        f"場合標籤：{'、'.join(metadata['tags']) or '未提供'}",
        f"氛圍關鍵字：{'、'.join(metadata['moods']) or '未提供'}",
    ])
    parts = [("基本資料", profile)] + [(name, sections[name]) for name in SECTION_NAMES]
    return [{"section": name, "text": f"{heading}｜{name}\n{body}"} for name, body in parts]


for chunk in build_chunks(sample_raw, sample_metadata)[:3]:
    print(chunk["text"], "\n" + "-" * 60)

## 8. 批次整理全部歌曲

每個片段整理成 `doc_id`、`text`、`metadata` 三個欄位（和老師的格式相同，只是單位從「一篇」變成「一個段落」）。程式會檢查歌曲編號、片段編號不重複，以及三個地區都有資料。

In [ ]:
songs, documents = [], []
for path in txt_paths:
    raw_text = path.read_text(encoding="utf-8-sig")
    if "�" in raw_text:
        raise ValueError(f"{path.name} 含有替代字元，請檢查編碼。")
    try:
        metadata = extract_metadata(raw_text, path.name)
        chunks = build_chunks(raw_text, metadata)
    except ValueError as exc:
        raise ValueError(f"處理失敗：{path.name}；原因：{exc}") from exc
    songs.append(metadata)
    for number, chunk in enumerate(chunks):
        documents.append({
            "doc_id": f"{metadata['song_id']}-{number}",
            "text": chunk["text"],
            "metadata": {**metadata, "section": chunk["section"]},
        })

song_ids = [song["song_id"] for song in songs]
index_doc_ids = [doc["doc_id"] for doc in documents]
if len(set(song_ids)) != len(song_ids) or len(set(index_doc_ids)) != len(index_doc_ids):
    raise ValueError("歌曲或片段編號重複，請檢查檔名。")
unknown_regions = {song["region"] for song in songs} - set(REGIONS)
if unknown_regions:
    raise ValueError(f"出現未定義的地區：{unknown_regions}")

display(pd.DataFrame([
    {"編號": s["song_id"], "地區": s["region"], "歌名": s["title"], "歌手": s["artist"],
     "年份": s["year_note"], "場合標籤": "、".join(s["tags"]), "來源數": len(s["source_urls"])}
    for s in songs
]))
print("歌曲數：", len(songs), "／片段數：", len(documents))
print("各地區歌曲數：", dict(Counter(song["region"] for song in songs)))
print("\n第一個片段的統一格式：")
print(json.dumps(documents[1], ensure_ascii=False, indent=2))

## 9. 📊 用 Matplotlib 看資料分布

用老師的固定套件畫三張圖：各地區歌曲數、發行年份分布、片段長度分布。（圖上用英文標籤，避免 Colab 預設字型無法顯示中文。）

In [ ]:
REGION_LABELS = {"歐美": "Western", "日本": "Japan", "華語": "Mandarin"}
REGION_COLORS = {"歐美": "#4C78A8", "日本": "#F58518", "華語": "#E45756"}

song_df = pd.DataFrame(songs)
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

region_counts = song_df["region"].value_counts().reindex(REGIONS)
axes[0].bar([REGION_LABELS[r] for r in REGIONS], region_counts.values, color=[REGION_COLORS[r] for r in REGIONS])
axes[0].set_title("Songs per region")

year_counts = song_df["year"].value_counts().sort_index()
axes[1].bar(year_counts.index.astype(str), year_counts.values, color="#72B7B2")
axes[1].set_title("Release year")
axes[1].tick_params(axis="x", rotation=45)

chunk_lengths = [len(doc["text"]) for doc in documents]
axes[2].hist(chunk_lengths, bins=20, color="#B279A2")
axes[2].set_title("Chunk length (characters)")

plt.tight_layout()
plt.show()
print("片段長度：最短", min(chunk_lengths), "／最長", max(chunk_lengths), "／平均", round(np.mean(chunk_lengths)))

## 10. 載入 EmbeddingGemma 2

和老師的程式相同：先查詢模型確切版本（commit），再載入該版本並記錄下來，讓 RAG02 載入一模一樣的模型。只啟用文字部分。

In [ ]:
try:
    MODEL_REVISION = HfApi().model_info(MODEL_ID, token=False).sha
    model = SentenceTransformer(
        MODEL_ID,
        revision=MODEL_REVISION,
        token=False,
        device=DEVICE,
        config_kwargs={"vision_config": None, "audio_config": None},
        model_kwargs={"dtype": DTYPE, "attn_implementation": "eager"},
    )
except Exception as exc:
    raise RuntimeError(
        "EmbeddingGemma 2 載入失敗。請確認網路、套件版本，必要時重新啟動工作階段。"
    ) from exc

print("模型：", MODEL_ID)
print("版本：", MODEL_REVISION)
print("裝置：", model.device, "／精度：", model.dtype)
print("向量維度：", model.get_sentence_embedding_dimension())

## 11. 組合送入模型的文字並檢查長度

文件用 `title: 歌名 - 歌手 | text: 片段`；問題用 `task: search result | query: 問題`。切塊之後每段都很短，遠低於 token 上限，不會被截斷。

In [ ]:
def document_title(metadata):
    return f"{metadata['title']} - {metadata['artist']}"


document_inputs = [
    DOCUMENT_TEMPLATE.format(title=document_title(doc["metadata"]), text=doc["text"])
    for doc in documents
]
TOKEN_LIMIT = min(2048, int(model.max_seq_length))


def count_input_tokens(text):
    return len(model.tokenizer(text, add_special_tokens=True, truncation=False)["input_ids"])


token_counts = [count_input_tokens(text) for text in document_inputs]
long_docs = [doc_id for doc_id, length in zip(index_doc_ids, token_counts) if length > TOKEN_LIMIT]
if long_docs:
    raise ValueError(f"以下片段超過 {TOKEN_LIMIT} tokens：{long_docs}")
print("token 上限：", TOKEN_LIMIT, "／最長片段：", max(token_counts), "tokens／平均：", round(np.mean(token_counts)))
print("第一筆真正送入 embedding 的文字：\n", document_inputs[1])

## 12. 產生片段向量

一個片段對應矩陣中的一列。`prompt=""` 表示前綴已自行加入；向量正規化後，內積就等於 cosine similarity。

In [ ]:
embeddings = model.encode(
    document_inputs,
    prompt="",
    batch_size=BATCH_SIZE,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True,
)
embeddings = np.ascontiguousarray(embeddings, dtype=np.float32)
if embeddings.ndim != 2 or embeddings.shape[0] != len(documents):
    raise ValueError("向量形狀與片段數不符。")
if not np.isfinite(embeddings).all() or np.any(np.linalg.norm(embeddings, axis=1) <= 0):
    raise ValueError("向量含有 NaN、無限值或零向量。")
faiss.normalize_L2(embeddings)
DIMENSION = int(embeddings.shape[1])
print("矩陣形狀（片段數, 向量維度）：", embeddings.shape)

## 13. 📊 把 216 個向量投影到 2D 看看

用 NumPy 的 SVD 做 PCA，把高維向量壓到 2 維。左圖依**地區**上色，右圖依**段落類型**上色，可以觀察語意相近的片段是否靠在一起（例如同語言、或同樣是「適合場合」的段落）。

In [ ]:
SECTION_LABELS = {
    "基本資料": "Profile", "創作背景": "Story", "氛圍與情緒": "Mood",
    "適合場合與用途": "Occasion", "歌手趣聞": "Artist", "延伸冷知識": "Trivia",
}
centered = embeddings - embeddings.mean(axis=0, keepdims=True)
_, _, vt = np.linalg.svd(centered, full_matrices=False)
coords = centered @ vt[:2].T

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
for region in REGIONS:
    mask = np.array([doc["metadata"]["region"] == region for doc in documents])
    axes[0].scatter(coords[mask, 0], coords[mask, 1], s=22, alpha=0.75,
                    color=REGION_COLORS[region], label=REGION_LABELS[region])
axes[0].set_title("Chunks by region (PCA)")
axes[0].legend()

for section, label in SECTION_LABELS.items():
    mask = np.array([doc["metadata"]["section"] == section for doc in documents])
    axes[1].scatter(coords[mask, 0], coords[mask, 1], s=22, alpha=0.75, label=label)
axes[1].set_title("Chunks by section (PCA)")
axes[1].legend()
plt.tight_layout()
plt.show()

## 14. 建立 FAISS 精確搜尋索引

`IndexFlatIP` 會和所有向量比較內積。FAISS 回傳的是向量位置，要透過 `index_doc_ids` 對回片段編號，再找到正文與 metadata。

In [ ]:
index = faiss.IndexFlatIP(DIMENSION)
index.add(embeddings)
documents_by_id = {doc["doc_id"]: doc for doc in documents}
assert index.ntotal == len(index_doc_ids) == len(documents)
print("索引類型：", type(index).__name__, "／向量數：", index.ntotal, "／維度：", index.d)

## 15. 試搜尋：看看真正取回了哪些片段

問題也用同一個模型向量化（使用查詢前綴）。這裡還沒有接 LLM，只看檢索結果是否合理。可以自己改 `TEST_QUESTIONS`。

**相似度不是正確率**：就算知識庫裡沒有答案，FAISS 仍然會回傳最相近的片段。

In [ ]:
def embed_question(question):
    if not isinstance(question, str) or not question.strip():
        raise ValueError("請輸入非空白問題。")
    query_input = QUERY_TEMPLATE.format(question=question.strip())
    if count_input_tokens(query_input) > TOKEN_LIMIT:
        raise ValueError("問題太長，請縮短。")
    query_vector = model.encode(
        [query_input], prompt="", convert_to_numpy=True,
        normalize_embeddings=True, show_progress_bar=False,
    )
    query_vector = np.ascontiguousarray(query_vector, dtype=np.float32)
    if query_vector.shape != (1, DIMENSION) or not np.isfinite(query_vector).all():
        raise ValueError("問題向量的維度或數值不正確。")
    faiss.normalize_L2(query_vector)
    return query_vector


def search_documents(question, top_k=3):
    if isinstance(top_k, bool) or not isinstance(top_k, (int, np.integer)) or top_k < 1:
        raise ValueError("top_k 必須是大於0的整數。")
    scores, positions = index.search(embed_question(question), min(int(top_k), index.ntotal))
    return [
        {"rank": rank, "score": float(score), **documents_by_id[index_doc_ids[int(position)]]}
        for rank, (position, score) in enumerate(zip(positions[0], scores[0]), start=1)
    ]


TEST_QUESTIONS = [
    "有什麼適合在婚禮上播放的歌？",
    "哪首歌是寫給因為疫情不能比賽的高中生？",
    "哪些歌是動畫的主題曲？",
    "BTS 的 Dynamite 在講什麼？",  # 知識庫沒有這首：觀察相似度與取回內容
]
TOP_K = 3
for question in TEST_QUESTIONS:
    print("\n問題：", question)
    display(pd.DataFrame([
        {"排名": r["rank"], "片段": r["doc_id"], "歌名": r["metadata"]["title"],
         "段落": r["metadata"]["section"], "相似度": round(r["score"], 4)}
        for r in search_documents(question, TOP_K)
    ]))

## 16. 保存索引、片段、編號對照與設定

和老師的範例一樣保存四個檔案；`config.json` 另外記錄切塊方式（`song_section_chunk`）與段落名稱，RAG02 會檢查這些設定。

In [ ]:
EXPORT_DIR = Path(tempfile.mkdtemp(prefix="rag_index_"))
faiss.write_index(index, str(EXPORT_DIR / "index.faiss"))
for filename, data in [("documents.json", documents), ("index_doc_ids.json", index_doc_ids)]:
    (EXPORT_DIR / filename).write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding="utf-8")

config = {
    "format_version": "1.0",
    "knowledge_base": "pop-song-stories",
    "created_at": datetime.now(ZoneInfo("Asia/Taipei")).isoformat(timespec="seconds"),
    "model_id": MODEL_ID,
    "model_revision": MODEL_REVISION,
    "model_config_kwargs": {"vision_config": None, "audio_config": None},
    "dimension": DIMENSION,
    "normalize_embeddings": True,
    "index_type": "IndexFlatIP",
    "metric": "cosine_similarity_via_normalized_inner_product",
    "document_template": DOCUMENT_TEMPLATE,
    "query_template": QUERY_TEMPLATE,
    "encode_prompt": "",
    "max_seq_length": TOKEN_LIMIT,
    "document_count": len(documents),
    "song_count": len(songs),
    "regions": REGIONS,
    "document_unit": "song_section_chunk",
    "chunk_sections": ["基本資料"] + SECTION_NAMES,
    "document_text_includes_title": True,
    "package_versions": PACKAGE_VERSIONS,
    "build_device": DEVICE,
    "build_dtype": str(DTYPE),
    "source_archive": archive_name,
    "source_archive_sha256": SOURCE_ARCHIVE_SHA256,
    "file_sha256": {
        name: hashlib.sha256((EXPORT_DIR / name).read_bytes()).hexdigest()
        for name in ["index.faiss", "documents.json", "index_doc_ids.json"]
    },
}
(EXPORT_DIR / "config.json").write_text(json.dumps(config, ensure_ascii=False, indent=2), encoding="utf-8")
for path in sorted(EXPORT_DIR.iterdir()):
    print(path.name, f"({path.stat().st_size:,} bytes)")

## 17. 重新載入並驗證

從剛保存的檔案重新載入，檢查雜湊、編號對照與維度，再用**同一個問題向量**比較保存前後的搜尋結果，確定 RAG02 拿到的資料是正確的。

In [ ]:
loaded_config = json.loads((EXPORT_DIR / "config.json").read_text(encoding="utf-8"))
for name, expected_hash in loaded_config["file_sha256"].items():
    assert hashlib.sha256((EXPORT_DIR / name).read_bytes()).hexdigest() == expected_hash, name

loaded_index = faiss.read_index(str(EXPORT_DIR / "index.faiss"))
loaded_documents = json.loads((EXPORT_DIR / "documents.json").read_text(encoding="utf-8"))
loaded_ids = json.loads((EXPORT_DIR / "index_doc_ids.json").read_text(encoding="utf-8"))
assert loaded_index.ntotal == len(loaded_documents) == len(loaded_ids) == loaded_config["document_count"]
assert loaded_index.d == loaded_config["dimension"]
assert loaded_documents == documents and loaded_ids == index_doc_ids

verification_query = embed_question(TEST_QUESTIONS[0])
k = min(TOP_K, index.ntotal)
before_scores, before_positions = index.search(verification_query, k)
after_scores, after_positions = loaded_index.search(verification_query, k)
np.testing.assert_array_equal(before_positions, after_positions)
np.testing.assert_allclose(before_scores, after_scores, atol=1e-6, rtol=1e-6)
print("通過：檔案完整性、片段對照、索引維度與保存前後一致。")

## 18. 打包成 `rag-data.zip` 並下載

In [ ]:
EXPORT_ZIP = EXPORT_DIR.parent / f"{EXPORT_DIR.name}_download" / EXPORT_ZIP_NAME
EXPORT_ZIP.parent.mkdir(parents=True, exist_ok=True)
required_files = ["index.faiss", "documents.json", "index_doc_ids.json", "config.json"]
with zipfile.ZipFile(EXPORT_ZIP, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for name in required_files:
        archive.write(EXPORT_DIR / name, arcname=name)
with zipfile.ZipFile(EXPORT_ZIP) as archive:
    assert archive.testzip() is None
    assert set(archive.namelist()) == set(required_files)
print("下載檔名：", EXPORT_ZIP_NAME, "／大小：", f"{EXPORT_ZIP.stat().st_size:,} bytes")

from google.colab import files
files.download(str(EXPORT_ZIP))

## 19.（選用）直接存到 Google 雲端硬碟

把 `SAVE_TO_DRIVE` 改成 `True`，會掛載雲端硬碟並把 `rag-data.zip` 存到「我的雲端硬碟」根目錄。

In [ ]:
SAVE_TO_DRIVE = False
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    drive_target = Path("/content/drive/MyDrive") / EXPORT_ZIP_NAME
    drive_target.write_bytes(EXPORT_ZIP.read_bytes())
    print("已存到雲端硬碟：", drive_target)
else:
    print("略過；需要時把 SAVE_TO_DRIVE 改成 True。")

## 20. 下一步：讓助教也能執行 RAG02

1. 把 `rag-data.zip` 上傳到 Google 雲端硬碟。
2. 對檔案按右鍵 →「共用」→ 一般存取權改成 **「知道連結的任何人」**。
3. 複製分享連結，貼到 RAG02 的 `RAG_DATA_URL`（一般分享連結即可，程式會自動轉成下載網址）。

RAG02 會自動下載並解壓縮 `rag-data.zip`，所以助教不需要手動上傳任何檔案。